In [1]:
from pyspark.sql import functions as F

df = spark.table("slv_flights_validated")
df.count()

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 3, Finished, Available, Finished, False)

597919

In [4]:
display(
    df.select(
        "flight_date",
        "reporting_airline",
        "flight_number",
        "cancelled",
        "cancellation_code",
        "dep_delay_minutes_signed",
        "dep_del15"
    ).limit(20)
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a70e867e-fc34-4681-ac18-0e437ec1f4c8)

In [7]:
df_status = (
    df
    .withColumn(
        "dep_status",
        F.when(
            F.col("cancelled") == True,
            F.lit("CANCELLED")
        )
        .when(
            F.col("dep_delay_minutes_signed").isNull(),
            F.lit("UNKNOWN")
        )
        .when(
            F.col("dep_delay_minutes_signed") <= 0,
            F.lit("EARLY_OR_ON_TIME")
        )
        .when(
            F.col("dep_delay_minutes_signed").between(1,14),
            F.lit("MINOR_DELAY")
        )
        .otherwise(
            F.lit("SIGNIFICANT_DELAY")
        )
    )
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 9, Finished, Available, Finished, False)

In [10]:
display(
    df_status
    .groupBy("dep_status")
    .count()
    .orderBy("dep_status")
)

df_status.count()


StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 12b15b1f-4841-4fa5-b1aa-b788c7008723)

597919

In [13]:
df_status = (
    df_status
    .withColumn(
        "cancellation_status",
        F.when(
            F.col("cancelled") == False,
            F.lit("NOT_CANCELLED")
        )
        .when(
            F.col("cancellation_code").isNull(),
            F.lit("Unknown_Cancel_Code")
        )
        .otherwise(
            F.col("cancellation_code")
        )
    )
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 15, Finished, Available, Finished, False)

In [14]:
display(
    df_status
    .groupBy(
        "cancelled",
        "cancellation_status"
    )
    .count()
    .orderBy(
        "cancelled",
        "cancellation_status"
    )
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72810be9-1d5b-4b34-9aab-d627b7283a75)

In [16]:
df_status = (
    df_status
    .withColumn(
        ("dep_delay_minutes_signed"),
        F.col("dep_delay_minutes_signed").cast("double")
    )
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 18, Finished, Available, Finished, False)

In [17]:
df_status.printSchema()

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 19, Finished, Available, Finished, False)

root
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- reporting_airline: string (nullable = true)
 |-- airline_dot_id: integer (nullable = true)
 |-- airline_iata_code: string (nullable = true)
 |-- tail_number: string (nullable = true)
 |-- flight_number: integer (nullable = true)
 |-- origin_airport_id: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- origin_city_name: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- dest_airport_id: integer (nullable = true)
 |-- dest: string (nullable = true)
 |-- dest_city_name: string (nullable = true)
 |-- dest_state: string (nullable = true)
 |-- crs_dep_time_hhmm: integer (nullable = true)
 |-- crs_dep_time_local: string (nullable = true)
 |-- dep_time_hhmm: integer (nullable = true)
 |-- dep_time_loca

In [20]:
original_rows = df.count()
transformed_rows = df_status.count()

print("Original: ", original_rows)
print("Transformed: ", transformed_rows)

assert original_rows == transformed_rows

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 22, Finished, Available, Finished, False)

Original:  597919
Transformed:  597919


In [23]:
sample3 = (
    df
    .select(
        "flight_key",
        "flight_date",
        "reporting_airline",
        "flight_number",
        "origin",
        "dest",
        "crs_dep_time_hhmm",
        "_bronze_ingested_at_utc",
        "_bronze_load_id",
        "_bronze_run_id"
    )
    .limit(3)
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 25, Finished, Available, Finished, False)

In [24]:
display(sample3)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2675d86d-2bb8-411a-8a16-ba046fa19acd)

In [25]:
one_row = sample3.limit(1)
practice_df = (
    sample3
    .unionByName(one_row)
    .unionByName(one_row)
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 27, Finished, Available, Finished, False)

In [26]:
practice_df.count()

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 28, Finished, Available, Finished, False)

5

In [31]:
from pyspark.sql.window import Window as Win

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 33, Finished, Available, Finished, False)

In [36]:
w = (
    Win
    .partitionBy("flight_key")
    .orderBy(
        F.col("_bronze_ingested_at_utc"),
        F.col("_bronze_load_id"),
        F.col("_bronze_run_id")
    )
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 38, Finished, Available, Finished, False)

In [37]:
ranked = (
    practice_df
    .withColumn(
        "duplicate_rank",
        F.row_number().over(w)
    )
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 39, Finished, Available, Finished, False)

In [38]:
display(
    ranked.select(
        "flight_key",
        "flight_number",
        "origin",
        "dest",
        "duplicate_rank"
    )
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 28cc1096-5eb0-4074-ba2e-9a65fe7d1510)

In [43]:
accepted = (
    ranked
    .filter(F.col("duplicate_rank") == 1)
)

duplicates = (
    ranked
    .filter(F.col("duplicate_rank") > 1)
)

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 45, Finished, Available, Finished, False)

In [45]:
input_rows = practice_df.count()
accepted_rows = accepted.count()
duplicate_rows = duplicates.count()

print("Input: ", input_rows)
print("Accepted: ", accepted_rows)
print("Duplicates: ", duplicate_rows)

assert input_rows == accepted_rows + duplicate_rows

StatementMeta(, 0acc54b6-ed9e-4edf-865c-6022ca8869a8, 47, Finished, Available, Finished, False)

Input:  5
Accepted:  3
Duplicates:  2
